In [0]:
/*
=============================================================================
PURPOSE: Date Dimension Table for Time Intelligence
=============================================================================
This view creates a comprehensive date dimension covering 120+ years 
(1980-2100) for consistent time-based analysis across all datasets.

TARGET SCHEMA: thekitchen.t (transformed data layer)
DATE RANGE: 1980-01-01 to 2100-12-31 (44,196 days)

BUSINESS VALUE:
- Enables standardized time-based filtering and grouping
- Supports year-over-year, quarter-over-quarter, and month-over-month comparisons
- Provides relative time offsets (e.g., "last 3 months", "current quarter")
- Essential for time series analysis and trending reports

KEY FEATURES:
- Period start dates (MonthDate, QuarterDate, YearDate) for easy aggregation
- Display formats (MMM for month names, DDD for day names)
- Sort keys for proper chronological ordering in reports
- Offset calculations relative to current date for dynamic filtering

USAGE PATTERN:
- Join to fact tables on Date column
- Filter using offset columns (e.g., WHERE MonthOffset BETWEEN -3 AND 0)
- Group by period dates for consistent aggregation boundaries
=============================================================================
*/

CREATE OR REPLACE VIEW t.Date AS
-- ============================================
-- Step 1: Define Date Range Parameters
-- ============================================
WITH params AS (
  SELECT
    to_date('1980-01-01') AS start_date,  -- Start of historical data coverage
    to_date('2100-12-31') AS end_date     -- Far future date for long-term planning
),
-- ============================================
-- Step 2: Generate Complete Date Sequence
-- ============================================
-- Creates one row per day using Spark's sequence() function
dates AS (
  SELECT
    explode(sequence(start_date, end_date, interval 1 day)) AS Date  -- Generate all dates in range
  FROM
    params
),
-- ============================================
-- Step 3: Calculate Date Attributes
-- ============================================
-- Derives all time-based attributes, sort keys, and helper keys for each date
base AS (
  SELECT
    Date,
    
    -- PERIOD START DATES (for aggregation boundaries)
    -- Use these for GROUP BY to ensure consistent period boundaries
    trunc(Date, 'MM') AS MonthDate,        -- First day of the month (e.g., 2024-03-15 → 2024-03-01)
    trunc(Date, 'Q') AS QuarterDate,       -- First day of the quarter (e.g., 2024-03-15 → 2024-01-01)
    trunc(Date, 'YEAR') AS YearDate,       -- First day of the year (e.g., 2024-03-15 → 2024-01-01)
    
    -- NUMERIC ATTRIBUTES (for filtering and calculations)
    year(Date) AS Year,                    -- Year as integer (e.g., 2024)
    month(Date) AS Month,                  -- Month as integer 1-12
    quarter(Date) AS Quarter,              -- Quarter as integer 1-4
    
    -- DISPLAY FORMATS (for user-friendly labels)
    date_format(Date, 'MMM') AS MMM,       -- Month abbreviation (e.g., "Jan", "Feb", "Mar")
    date_format(Date, 'EEE') AS DDD,       -- Day name abbreviation (e.g., "Mon", "Tue", "Wed")
    
    -- SORT KEYS (for proper chronological ordering in reports)
    -- These ensure correct sorting when displaying periods
    year(Date) AS YearSortKey,                              -- 2024
    (year(Date) * 100 + month(Date)) AS YearMonthSortKey,   -- 202403 (March 2024)
    (year(Date) * 10 + quarter(Date)) AS YearQuarterSortKey,-- 20241 (Q1 2024)
    
    -- HELPER KEYS (internal use for offset calculations)
    -- These enable efficient relative time calculations
    (year(Date) * 12 + month(Date)) AS YearMonthKey,        -- Continuous month counter
    (year(Date) * 4 + quarter(Date)) AS YearQuarterKey      -- Continuous quarter counter
  FROM
    dates
),
-- ============================================
-- Step 4: Capture Current Date Context
-- ============================================
-- Stores current period information for calculating relative time offsets
-- This enables dynamic filtering like "last 3 months" or "current quarter"
curr AS (
  SELECT
    year(current_date()) AS CurrYear,                                                 -- Current year
    (year(current_date()) * 12 + month(current_date())) AS CurrYearMonthKey,         -- Current month key
    (year(current_date()) * 4 + quarter(current_date())) AS CurrYearQuarterKey       -- Current quarter key
)
-- ============================================
-- Step 5: Final Output with Offset Calculations
-- ============================================
-- Combines all date attributes with relative time offsets for dynamic filtering
SELECT
  b.Date,                                    -- Primary date key for joins
  
  -- PERIOD START DATES (for GROUP BY aggregations)
  b.MonthDate,                               -- First day of month
  b.QuarterDate,                             -- First day of quarter
  b.YearDate,                                -- First day of year
  
  -- DISPLAY ATTRIBUTES (for labels in reports/dashboards)
  b.MMM,                                     -- Month name abbreviation ("Jan", "Feb", etc.)
  b.DDD,                                     -- Day name abbreviation ("Mon", "Tue", etc.)
  
  -- SORT KEYS (for chronological ordering)
  b.YearSortKey,                             -- Sort by year
  b.YearQuarterSortKey,                      -- Sort by year-quarter
  b.YearMonthSortKey,                        -- Sort by year-month
  
  -- RELATIVE TIME OFFSETS (for dynamic filtering)
  -- Negative = past, 0 = current, Positive = future
  (b.Year - c.CurrYear) AS YearOffset,                             -- Years from today (e.g., -1 = last year)
  (b.YearQuarterKey - c.CurrYearQuarterKey) AS QuarterOffset,      -- Quarters from today (e.g., -4 = 4 quarters ago)
  (b.YearMonthKey - c.CurrYearMonthKey) AS MonthOffset             -- Months from today (e.g., -3 = 3 months ago)
FROM
  base b 
  CROSS JOIN curr c;  -- Cross join to apply current date context to all rows

/*
=============================================================================
USAGE EXAMPLES:
=============================================================================

-- Example 1: Get last 3 months of data
SELECT * 
FROM fact_table f
JOIN t.Date d ON f.transaction_date = d.Date
WHERE d.MonthOffset BETWEEN -3 AND 0;

-- Example 2: Year-over-year comparison (current month vs same month last year)
SELECT 
  d.MMM,
  d.Year,
  SUM(f.amount) as total_amount
FROM fact_table f
JOIN t.Date d ON f.transaction_date = d.Date
WHERE d.MonthOffset IN (0, -12)  -- Current month and 12 months ago
GROUP BY d.MMM, d.Year
ORDER BY d.YearMonthSortKey;

-- Example 3: Current quarter aggregation
SELECT 
  d.QuarterDate,
  COUNT(*) as transaction_count
FROM fact_table f
JOIN t.Date d ON f.transaction_date = d.Date
WHERE d.QuarterOffset = 0  -- Current quarter only
GROUP BY d.QuarterDate;

-- Example 4: Trending over time with proper ordering
SELECT 
  d.MonthDate,
  d.MMM || ' ' || d.Year as period_label,
  AVG(f.value) as avg_value
FROM fact_table f
JOIN t.Date d ON f.transaction_date = d.Date
WHERE d.MonthOffset BETWEEN -12 AND 0  -- Last 12 months
GROUP BY d.MonthDate, d.MMM, d.Year, d.YearMonthSortKey
ORDER BY d.YearMonthSortKey;
=============================================================================
*/